# 05 - Heading-aware chunking

Heading-aware chunking preserves section meaning. It detects likely headings and attaches the current heading to the chunk text.

In RAG, this matters because a paragraph may say "this criterion" or "the appeal". The heading can tell the retriever and the model what that text is about.

In [ ]:
from pathlib import Path
import re

from pypdf import PdfReader

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/sample_documents/AUG242026_05B2203_AAO.pdf").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the Awesome_RAG checkout")
PDF_PATH = ROOT / "data" / "sample_documents" / "AUG242026_05B2203_AAO.pdf"

def normalize_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def load_pdf_pages(path):
    reader = PdfReader(str(path))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = normalize_text(page.extract_text() or "")
        if text:
            pages.append({"page": page_number, "text": text})
    return pages

def chunk_stats(chunks):
    lengths = [len(chunk["text"].split()) for chunk in chunks]
    return {"chunks": len(chunks), "min_words": min(lengths), "avg_words": round(sum(lengths) / len(lengths), 1), "max_words": max(lengths)}

def print_chunks(chunks, limit=6, preview_chars=650):
    for chunk in chunks[:limit]:
        print(f"{chunk['id']} | page {chunk['page_start']}-{chunk['page_end']} | heading: {chunk['heading']}")
        print(chunk["text"][:preview_chars])
        print("-" * 100)
    if len(chunks) > limit:
        print(f"... {len(chunks) - limit} more chunks")

pages = load_pdf_pages(PDF_PATH)
{"pages": len(pages), "words": sum(len(page["text"].split()) for page in pages)}

## Heading-aware chunking function

This is a teaching version of heading detection. Real systems should tune heading rules to the document family.

`looks_like_heading()` checks whether a line is short and mostly uppercase, or looks like a numbered/label heading. `heading_aware_chunks()` keeps track of the most recent heading and prepends it to each chunk.

This gives each chunk stronger context for retrieval.

In [ ]:
def looks_like_heading(line):
    """Heuristic heading detector for extracted PDF text."""
    stripped = line.strip()
    if not stripped or len(stripped) > 90:
        return False

    words = stripped.split()
    if len(words) > 12:
        return False

    letters = [char for char in stripped if char.isalpha()]
    uppercase_ratio = sum(char.isupper() for char in letters) / max(1, len(letters))

    return uppercase_ratio > 0.65 or bool(re.match(r"^(\d+\.|[IVX]+\.|[A-Z][A-Za-z]+:)", stripped))


def heading_aware_chunks(pages, max_words=220):
    """Create chunks that include the most recent detected heading."""
    chunks = []

    for page in pages:
        heading = f"Page {page['page']}"
        current_lines = []

        def save_current_chunk():
            if current_lines:
                body = "\n".join(current_lines).strip()
                chunks.append({
                    "id": f"section-{len(chunks) + 1}",
                    "page_start": page["page"],
                    "page_end": page["page"],
                    "heading": heading,
                    "text": f"{heading}\n{body}",
                })

        for line in page["text"].splitlines():
            stripped = line.strip()
            if not stripped:
                continue

            if looks_like_heading(stripped):
                save_current_chunk()
                heading = stripped
                current_lines = []
            else:
                current_lines.append(stripped)
                if len(" ".join(current_lines).split()) >= max_words:
                    save_current_chunk()
                    current_lines = []

        save_current_chunk()

    return chunks

chunks = heading_aware_chunks(pages, max_words=220)
chunk_stats(chunks)

In [ ]:
print_chunks(chunks, limit=8)

## How to judge it

Heading-aware chunking is strong when the document has meaningful sections. It is weak when heading detection is noisy or the PDF extraction loses structure.

Always inspect detected headings before trusting this method in a production RAG system.